# **Chapter 2: Build A Transformer**

This is for the book Build a Text to Image Generator from Scratch by Mark Liu, published by Manning Publications. See https://mng.bz/vZem and https://github.com/markhliu for details.

Be sure to set the runtime type of this colab notebook to GPU

In [1]:
# clone the book's GitHub repository
!git clone https://github.com/markhliu/txt2img

Cloning into 'txt2img'...
remote: Enumerating objects: 369, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 369 (delta 47), reused 10 (delta 10), pack-reused 281 (from 2)
Receiving objects: 100% (369/369), 115.88 MiB | 30.01 MiB/s, done.
Resolving deltas: 100% (155/155), done.


In [2]:
# allow access to local modules and files in the book's repository
import sys
sys.path.append("/content/txt2img")

In [3]:
import requests, os, tarfile

url=("https://raw.githubusercontent.com/neychev/"
     "small_DL_repo/master/datasets/Multi30k/training.tar.gz")  #A
os.makedirs("files", exist_ok=True)
if not os.path.exists("files/training.tar.gz"):  #B
    fb1=requests.get(url)
    with open("files/training.tar.gz","wb") as f:
        f.write(fb1.content)
train=tarfile.open('files/training.tar.gz')  #C
train.extractall('files')  #D
train.close()
#A The URL to download the training dataset
#B Download the dataset to your computer
#C Unzip the file
#D Place content in the /files/ folder


In [4]:
with open("files/train.de", 'rb') as fb:
    trainde = fb.readlines()
with open("files/train.en", 'rb') as fb:
    trainen = fb.readlines()
trainde=[i.decode("utf-8").strip() for i in trainde]
trainen=[i.decode("utf-8").strip() for i in trainen]


In [5]:
from pprint import pprint

print(f"the length of the list trainde is {len(trainde)}")
print(f"the length of the list trainen is {len(trainen)}")
print(f"the first five elements of the list trainde are")
pprint(trainde[:5])
print(f"the first five elements of the list trainen are")
pprint(trainen[:5])


the length of the list trainde is 29001
the length of the list trainen is 29001
the first five elements of the list trainde are
['Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.',
 'Ein kleines Mädchen klettert in ein Spielhaus aus Holz.',
 'Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.',
 'Zwei Männer stehen am Herd und bereiten Essen zu.']
the first five elements of the list trainen are
['Two young, White males are outside near many bushes.',
 'Several men in hard hats are operating a giant pulley system.',
 'A little girl climbing into a wooden playhouse.',
 'A man in a blue shirt is standing on a ladder cleaning a window.',
 'Two men are at the stove preparing food.']


In [6]:
import os, spacy

try:
    de_tokenizer = spacy.load("de_core_news_sm")  #A
except IOError:
    os.system("python -m spacy download de_core_news_sm")  #B
    de_tokenizer = spacy.load("de_core_news_sm")
try:
    en_tokenizer = spacy.load("en_core_web_sm")  #C
except IOError:
    os.system("python -m spacy download en_core_web_sm")  #D
    en_tokenizer = spacy.load("en_core_web_sm")
#A Try to load the German model
#B If the German model is not found, download it to your computer
#C Try to load the English model
#D If the English model is not found, download it to your computer


In [7]:
tokenized_de= [tok.text for tok in
              de_tokenizer.tokenizer(trainde[0])]
tokenized_en=[tok.text for tok in
              en_tokenizer.tokenizer(trainen[0])]
print(tokenized_de)
print(tokenized_en)


['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'vieler', 'Büsche', '.']
['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


In [8]:
from collections import Counter

en_tokens=[["BOS"]+[tok.text for tok in en_tokenizer.tokenizer(x)]
           +["EOS"] for x in trainen]  #A
PAD=0
UNK=1
word_count=Counter()
for sentence in en_tokens:
    for word in sentence:
        word_count[word]+=1
frequency=word_count.most_common(50000)
total_en_words=len(frequency)+2
# a dictionary mapping tokens to indexes
en_word_dict={w[0]:idx+2 for idx,w in enumerate(frequency)}  #B
en_word_dict["PAD"]=PAD
en_word_dict["UNK"]=UNK  #C
# another dictionary to map indexes to tokens
en_idx_dict={v:k for k,v in en_word_dict.items()}  #D
#A Add BOS and EOS at the beginning and end of each phrase
#B Assign an index to each unique token
#C The padding token and unknown tokens are assigned indexes 0 and 1, respectively
#D A dictionary to map indexes back to tokens


In [9]:
enidx=[en_word_dict.get(i,UNK) for i in tokenized_en]
print(enidx)


[19, 25, 15, 1165, 804, 17, 57, 84, 334, 1329, 5]


In [10]:
entokens=[en_idx_dict.get(i,"UNK") for i in enidx]
print(entokens)
en_phrase=" ".join(entokens)

# removing spaces before characters like '?'
for x in '''?:;.,'("-!&)%''':
    en_phrase=en_phrase.replace(f" {x}",f"{x}")
print(en_phrase)


['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']
Two young, White males are outside near many bushes.


In [11]:
UNK

1

In [12]:
de_tokens= [["BOS"]+[tok.text for tok in de_tokenizer.tokenizer(x)]
           +["EOS"] for x in trainde]  #A
de_word_count=Counter()
for sentence in de_tokens:
    for word in sentence:
        de_word_count[word]+=1
defrequency=de_word_count.most_common(50000)
total_de_words=len(defrequency)+2
de_word_dict={w[0]:idx+2 for idx,w in enumerate(defrequency)}  #B
de_word_dict["PAD"]=PAD
de_word_dict["UNK"]=UNK  #C
de_idx_dict={v:k for k,v in de_word_dict.items()}  #D
#A Add BOS and EOS at the beginning and end of each phrase
#B Assign an index to each unique token
#C The padding token and unknown tokens are assigned indexes 0 and 1, respectively
#D A dictionary to map indexes back to tokens


In [13]:
deidx=[de_word_dict.get(i,UNK) for i in tokenized_de]
print(deidx)


[21, 85, 257, 31, 87, 22, 94, 7, 16, 112, 5497, 3161, 4]


In [14]:
detokens=[de_idx_dict.get(i,"UNK") for i in deidx]
print(detokens)
de_phrase=" ".join(detokens)
for x in '''?:;.,'("-!&)%''':
    de_phrase=de_phrase.replace(f" {x}",f"{x}")
print(de_phrase)


['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'vieler', 'Büsche', '.']
Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.


In [15]:
out_en_ids=[[en_word_dict.get(w,UNK) for w in s]
            for s in en_tokens]
out_de_ids=[[de_word_dict.get(w,UNK) for w in s]
            for s in de_tokens]
sorted_ids=sorted(range(len(out_de_ids)),
                  key=lambda x:len(out_de_ids[x]))
out_de_ids=[out_de_ids[x] for x in sorted_ids]
out_en_ids=[out_en_ids[x] for x in sorted_ids]


In [16]:
import numpy as np

batch_size=128
idx_list=np.arange(0,len(de_tokens),batch_size)
np.random.shuffle(idx_list)

batch_indexs=[]
for idx in idx_list:
    batch_indexs.append(np.arange(idx,min(len(de_tokens),
                                          idx+batch_size)))


In [17]:
def seq_padding(X, padding=PAD):
    L = [len(x) for x in X]
    ML = max(L)
    padded_seq = np.array([np.concatenate([x,
                   [padding] * (ML - len(x))])
        if len(x) < ML else x for x in X])
    return padded_seq


In [18]:
class Batch:
    def __init__(self, src, trg=None, pad=0):
        src = torch.from_numpy(src).to(DEVICE).long()
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2)  #A
        if trg is not None:
            trg = torch.from_numpy(trg).to(DEVICE).long()
            self.trg = trg[:, :-1]  #B
            self.trg_y = trg[:, 1:]  #C
            self.trg_mask = make_std_mask(self.trg, pad)  #D
            self.ntokens = (self.trg_y != pad).data.sum()
#A Create a source mask to hide padding at the end of the sentence
#B Create input to the decoder
#C Shift the input one token to the right and use it as output
#D Create a target mask


In [19]:
from utils.transformer_util import Batch

batches=[]
for b in batch_indexs:
    batch_en=[out_en_ids[x] for x in b]
    batch_de=[out_de_ids[x] for x in b]
    batch_en=seq_padding(batch_en)
    batch_de=seq_padding(batch_de)
    batches.append(Batch(batch_de,batch_en))


In [20]:
src_vocab = len(de_word_dict)
tgt_vocab = len(en_word_dict)
print(f"there are {src_vocab} distinct German tokens")
print(f"there are {tgt_vocab} distinct English tokens")


there are 19214 distinct German tokens
there are 10837 distinct English tokens


In [21]:
from torch import nn
import math
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model
    def forward(self, x):
        out = self.lut(x) * math.sqrt(self.d_model)
        return out


In [22]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):  #A
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=DEVICE)
        position = torch.arange(0., max_len,
                                device=DEVICE).unsqueeze(1)
        div_term = torch.exp(torch.arange(
            0., d_model, 2, device=DEVICE)
            * -(math.log(10000.0) / d_model))
        pe_pos = torch.mul(position, div_term)
        pe[:, 0::2] = torch.sin(pe_pos)  #B
        pe[:, 1::2] = torch.cos(pe_pos)  #C
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)].requires_grad_(False)  #D
        out = self.dropout(x)
        return out
#A Initiate the class, allowing a maximum of 5000 positions
#B Apply sine function to even indexes in the vector
#C Apply cosine function to odd indexes in the vector
#D Add positional encoding to word embedding


In [23]:
def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query,
              key.transpose(-2, -1)) / math.sqrt(d_k)  #A
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)  #B
    p_attn = nn.functional.softmax(scores, dim=-1)  #C
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn  #D
#A Scaled attention score is the dot product of query and key, scaled by the square root of dk
#B If there is a mask, hide future elements in the sequence
#C Calculate attention weights
#D Return both attention and attention weights


In [24]:
class Transformer(nn.Module):
    def __init__(self, encoder, decoder,
                 src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder  #A
        self.decoder = decoder  #B
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator
    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)
    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt),
                            memory, src_mask, tgt_mask)
    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask)  #C
        output = self.decode(memory, src_mask, tgt, tgt_mask)  #D
        return output
#A Define an encoder in the transformer
#B Define a decoder in the transformer
#C Source language is encoded into an abstract vector representation by the encoder
#D The decoder uses the vector representation to generate the translation in the target language


In [25]:
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList(
            [deepcopy(layer) for i in range(N)])
        self.norm = LayerNorm(layer.size)
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
            output = self.norm(x)
        return output


In [26]:
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn,
                 feed_forward, dropout):
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = nn.ModuleList([deepcopy(
        SublayerConnection(size, dropout)) for i in range(3)])
    def forward(self, x, memory, src_mask, tgt_mask):
        x = self.sublayer[0](x, lambda x:
                 self.self_attn(x, x, x, tgt_mask))  #A
        x = self.sublayer[1](x, lambda x:
                 self.src_attn(x, memory, memory, src_mask))  #B
        output = self.sublayer[2](x, self.feed_forward)  #C
        return output
#A The first sublayer is a masked multi-head attention layer
#B The second sublayer is a cross-attention layer between the two languages
#C The third sublayer is a feed-forward network


In [27]:
def create_model(src_vocab, tgt_vocab, N, d_model,
                 d_ff, h, dropout=0.1):
    attn=MultiHeadedAttention(h, d_model).to(DEVICE)
    ff=PositionwiseFeedForward(d_model, d_ff, dropout).to(DEVICE)
    pos=PositionalEncoding(d_model, dropout).to(DEVICE)
    model = Transformer(
        Encoder(EncoderLayer(d_model,deepcopy(attn),deepcopy(ff),
                             dropout).to(DEVICE),N).to(DEVICE),  #A
        Decoder(DecoderLayer(d_model,deepcopy(attn),
             deepcopy(attn),deepcopy(ff), dropout).to(DEVICE),
                N).to(DEVICE),  #B
        nn.Sequential(Embeddings(d_model, src_vocab).to(DEVICE),
                      deepcopy(pos)),  #C
        nn.Sequential(Embeddings(d_model, tgt_vocab).to(DEVICE),
                      deepcopy(pos)),  #D
        Generator(d_model, tgt_vocab)).to(DEVICE)  #E
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model.to(DEVICE)
#A Create an encoder by instantiating the Encoder() class
#B Create a decoder by instantiating the Decoder() class
#C Create src_embed by generating input embeddings for the source language
#D Create tgt_embed by generating input embeddings for the target language
#E Create a generator by instantiating the Generator() class


In [28]:
from utils.transformer_util import create_model

model = create_model(src_vocab, tgt_vocab, N=6,
    d_model=256, d_ff=1024, h=8, dropout=0.1)


In [29]:
from utils.transformer_util import (NoamOpt, LabelSmoothing,
       SimpleLossCompute)
import torch

optimizer = NoamOpt(256, 1, 2000, torch.optim.Adam(
    model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))
criterion = LabelSmoothing(tgt_vocab,
                           padding_idx=0, smoothing=0.0)
loss_func = SimpleLossCompute(
            model.generator, criterion, optimizer)


In [ ]:
# I didn't finish the training, but you can if you want
# or you can download the trained weights (go to the next code cell)
for epoch in range(50):
    model.train()
    tloss=0
    tokens=0
    for batch in batches:
        out = model(batch.src, batch.trg,
                    batch.src_mask, batch.trg_mask)  #A
        loss = loss_func(out, batch.trg_y, batch.ntokens)  #B
        tloss += loss
        tokens += batch.ntokens  #C
    print(f"Epoch {epoch}, average loss: {tloss/tokens}")
torch.save(model.state_dict(),"/content/txt2img/files/de2en.pth")  #D
#A Predict the next token using the transformer
#B Calculate loss and adjust model parameters
#C Count the number of tokens in the batch
#D Save the weights in the trained model after training


Epoch 0, average loss: 7.363134860992432


In [ ]:
from utils.transformer_util import subsequent_mask
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
def de2en(ger):
    tokenized_ger= [tok.text for tok in de_tokenizer.tokenizer(ger)]
    tokenized_ger=["BOS"]+tokenized_ger+["EOS"]
    geridx=[de_word_dict.get(i,UNK) for i in tokenized_ger]
    src=torch.tensor(geridx).long().to(DEVICE).unsqueeze(0)
    src_mask=(src!=0).unsqueeze(-2)
    memory=model.encode(src,src_mask)  #A
    start_symbol=en_word_dict["BOS"]
    ys = torch.ones(1, 1).fill_(start_symbol).type_as(src.data)
    translation=[]
    for i in range(100):
        out = model.decode(memory,src_mask,ys,
        subsequent_mask(ys.size(1)).type_as(src.data))
        prob = model.generator(out[:, -1])  #B
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, torch.ones(1, 1).type_as(
            src.data).fill_(next_word)], dim=1)
        sym = en_idx_dict[ys[0, -1].item()]
        if sym != 'EOS':  #C
            translation.append(sym)
        else:
            break
    trans=" ".join(translation)
    for x in '''?:;.,'("-!&)%''':
        trans=trans.replace(f" {x}",f"{x}")  #D
    return trans
#A Use encoder to convert German to vector representations
#B Predict the next English token using the decoder
#C Stops translating when the next token is EOS
#D Join the predicted tokens to form an English sentence as the translation


In [ ]:

model.eval()
for i in range(5):
    print("original Ger:", trainde[100+i])
    print("original Eng:", trainen[100+i])
    print("translated Eng:", de2en(trainde[100+i]))
